Läser in alla regiondata för rumsbeläggning och gästnätter. Gör lite feature engineering

In [2]:
import pandas as pd

In [3]:
from pathlib import Path

print(Path.cwd())

C:\Users\ES7233\emil_dev\AppliedAI\tourism_weather


In [4]:
from pathlib import Path

DATA_DIR = Path("/Users/ES7233/emil_dev/AppliedAI/tourism_weather/data")

In [5]:

def load_folder_simple(folder_path, value_name):
   
    dfs = []

    for file in Path(folder_path).glob("*.xlsx"):
        df = pd.read_excel(file, skiprows=4)
        df.columns = ["month_name", "year", value_name]
        df["region"] = file.stem
        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)

In [6]:
guest_df = load_folder_simple(DATA_DIR / "guest_nights", "guest_nights")
occ_df = load_folder_simple(DATA_DIR / "occupancy", "occupancy_rate")

C:\Users\ES7233\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\ES7233\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\ES7233\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\ES7233\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openp

In [7]:
month_map = {
    "Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4,
    "Maj": 5, "Jun": 6, "Jul": 7, "Aug": 8,
    "Sep": 9, "Okt": 10, "Nov": 11, "Dec": 12
}

def add_date(df):
    df = df.copy()

    df["month"] = df["month_name"].map(month_map)
    df["year"] = pd.to_numeric(df["year"], errors="coerce")

    df = df.dropna(subset=["year", "month"])

    df["date"] = pd.to_datetime(
        dict(
            year=df["year"].astype(int),
            month=df["month"].astype(int),
            day=1
        )
    )

    return df

In [8]:
guest_df = add_date(guest_df)
occ_df = add_date(occ_df)

In [9]:
print(guest_df.columns)
print(occ_df.columns)

Index(['month_name', 'year', 'guest_nights', 'region', 'month', 'date'], dtype='str')
Index(['month_name', 'year', 'occupancy_rate', 'region', 'month', 'date'], dtype='str')


In [10]:
df_t= guest_df.merge(
    occ_df,
    on=["region", "date"],
    how="inner"
)

In [11]:
df_t.info()

<class 'pandas.DataFrame'>
RangeIndex: 1230 entries, 0 to 1229
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   month_name_x    1230 non-null   str           
 1   year_x          1230 non-null   float64       
 2   guest_nights    1230 non-null   object        
 3   region          1230 non-null   str           
 4   month_x         1230 non-null   float64       
 5   date            1230 non-null   datetime64[us]
 6   month_name_y    1230 non-null   str           
 7   year_y          1230 non-null   float64       
 8   occupancy_rate  1230 non-null   object        
 9   month_y         1230 non-null   float64       
dtypes: datetime64[us](1), float64(4), object(2), str(3)
memory usage: 115.1+ KB


In [12]:
df_t = df_t.rename(columns={
    "year_x": "year",
    "month_x": "month"
})

df_t = df_t.drop(columns=[
    "month_name_x",
    "month_name_y",
    "year_y",
    "month_y"
])

In [13]:
df_t["guest_nights"] = pd.to_numeric(df_t["guest_nights"], errors="coerce")
df_t["occupancy_rate"] = pd.to_numeric(df_t["occupancy_rate"], errors="coerce")

In [14]:
df_t = df_t.dropna().sort_values(["region", "date"]).reset_index(drop=True)

In [15]:
df_t.info()
df_t.head()

<class 'pandas.DataFrame'>
RangeIndex: 1230 entries, 0 to 1229
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   year            1230 non-null   float64       
 1   guest_nights    1230 non-null   int64         
 2   region          1230 non-null   str           
 3   month           1230 non-null   float64       
 4   date            1230 non-null   datetime64[us]
 5   occupancy_rate  1230 non-null   float64       
dtypes: datetime64[us](1), float64(3), int64(1), str(1)
memory usage: 69.1 KB


,year,guest_nights,region,month,date,occupancy_rate
0,2015.0,815341,dalarna,7.0,2015-07-01,0.554058
1,2015.0,459539,dalarna,8.0,2015-08-01,0.427013
2,2015.0,172431,dalarna,9.0,2015-09-01,0.327624
3,2015.0,154394,dalarna,10.0,2015-10-01,0.285888
4,2015.0,127696,dalarna,11.0,2015-11-01,0.280259


In [16]:
df_t["year"] = df_t["year"].astype(int)
df_t["month"] = df_t["month"].astype(int)

In [17]:
df_t.groupby("region").size()

region
dalarna           129
gotland           129
jamtland          129
norrbotten        129
riket              69
skane             129
stockholm         129
vasterbotten      129
vasternorrland    129
vastragotaland    129
dtype: int64

In [18]:
df_t["guest_lag1"] = df_t.groupby("region")["guest_nights"].shift(1)
df_t["guest_lag2"] = df_t.groupby("region")["guest_nights"].shift(2)
df_t["guest_lag12"] = df_t.groupby("region")["guest_nights"].shift(12)

df_t["occ_lag1"] = df_t.groupby("region")["occupancy_rate"].shift(1)

In [19]:
df_t[["region", "date", "guest_nights", "guest_lag1"]].head()

,region,date,guest_nights,guest_lag1
0,dalarna,2015-07-01,815341,NaN
1,dalarna,2015-08-01,459539,815341.0
2,dalarna,2015-09-01,172431,459539.0
3,dalarna,2015-10-01,154394,172431.0
4,dalarna,2015-11-01,127696,154394.0


In [20]:
df_t.isna().sum()

year                0
guest_nights        0
region              0
month               0
date                0
occupancy_rate      0
guest_lag1         10
guest_lag2         20
guest_lag12       120
occ_lag1           10
dtype: int64

In [21]:
import pandas as pd
import re
from pathlib import Path

def load_smhi_temp(path, region):
    rows = []

    with open(path, encoding="utf-8-sig") as f:
        for line in f:
            parts = line.strip().split(";")

            for i, value in enumerate(parts):
                value = value.strip()

                # hittar månad, t.ex. 2015-07
                if re.match(r"^\d{4}-\d{2}$", value):
                    if i + 1 < len(parts):
                        temp = parts[i + 1].replace(",", ".")
                        rows.append({
                            "region": region,
                            "date": pd.to_datetime(value),
                            "temp_mean": pd.to_numeric(temp, errors="coerce")
                        })

    weather = pd.DataFrame(rows)
    weather = weather.dropna().reset_index(drop=True)

    return weather

In [22]:
from pathlib import Path

weather_parts = []

for file in Path("data/smhi/temp").glob("*.csv"):
    region = file.stem
    weather_parts.append(load_smhi_temp(file, region))

weather_df = pd.concat(weather_parts, ignore_index=True)

In [23]:
weather_df["region"].unique()
weather_df.groupby("region").size()

region
dalarna           684
gotland           964
jamtland          983
lulea             973
skane             507
stockholm         351
sundsvall         939
umea              731
vastragotaland    432
dtype: int64

In [24]:
df_t["region"].unique()

<ArrowStringArray>
[       'dalarna',        'gotland',       'jamtland',     'norrbotten',
          'riket',          'skane',      'stockholm',   'vasterbotten',
 'vasternorrland', 'vastragotaland']
Length: 10, dtype: str

In [25]:
weather_df["region"] = weather_df["region"].replace({
    "lulea": "norrbotten",
    "umea": "vasterbotten",
    "sundsvall": "vasternorrland"
})

In [26]:
set(df_t["region"].unique()) - set(weather_df["region"].unique())

{'riket'}

In [26]:
df_final = df_t.merge(
    weather_df,
    on=["region", "date"],
    how="left"
)

In [27]:
df_final.groupby("region")["temp_mean"].apply(lambda x: x.isna().sum())

region
dalarna            2
gotland            2
jamtland           2
norrbotten         2
riket             69
skane              7
stockholm          2
vasterbotten       2
vasternorrland     2
vastragotaland     4
Name: temp_mean, dtype: int64

In [28]:
df_model = df_final.dropna().reset_index(drop=True)

In [29]:
def load_smhi_daily_rain(path, region):
    rows = []

    with open(path, encoding="utf-8-sig") as f:
        for line in f:
            parts = [p.strip() for p in line.strip().split(";")]

            # SMHI regnfil:
            # 0 = från datetime
            # 1 = till datetime
            # 2 = representativt dygn
            # 3 = nederbörd
            if len(parts) >= 4:
                date = pd.to_datetime(parts[2], errors="coerce")
                rain = pd.to_numeric(parts[3].replace(",", "."), errors="coerce")

                if pd.notna(date) and pd.notna(rain):
                    rows.append({
                        "region": region,
                        "date": date,
                        "rain_mm": rain
                    })

    return pd.DataFrame(rows).reset_index(drop=True)

Aggregera regn från dag, till månad.

In [30]:
from pathlib import Path

rain_parts = []

for file in Path("data/smhi/regn").glob("*.csv"):
    region = file.stem
    rain_parts.append(load_smhi_daily_rain(file, region))

rain_df = pd.concat(rain_parts, ignore_index=True)

In [31]:
rain_df.head()
rain_df.groupby("region").size()

region
dalarna           60360
gotland           23427
jamtland          13825
norrbotten        25120
skane             11022
stockholm         26749
vasterbotten      19968
vasternorrland    19878
vastragotaland    16438
dtype: int64

In [32]:
rain_df["month"] = rain_df["date"].dt.to_period("M")

In [33]:
rain_df["month"] = rain_df["date"].dt.to_period("M")

rain_monthly = (
    rain_df
    .groupby(["region", "month"])
    .agg(
        rain_sum=("rain_mm", "sum"),
        rain_days=("rain_mm", lambda x: (x > 1).sum()),
        rain_mean=("rain_mm", "mean")
    )
    .reset_index()
)

rain_monthly["date"] = rain_monthly["month"].dt.to_timestamp()
rain_monthly = rain_monthly.drop(columns="month")

In [34]:
rain_monthly.head()

,region,rain_sum,rain_days,rain_mean,date
0,dalarna,98.4,12,3.174194,1860-01-01
1,dalarna,13.5,3,0.465517,1860-02-01
2,dalarna,27.1,5,0.874194,1860-03-01
3,dalarna,63.4,10,2.113333,1860-04-01
4,dalarna,55.3,8,1.783871,1860-05-01


In [35]:
rain_monthly = rain_monthly[
    (rain_monthly["date"] >= df_t["date"].min()) &
    (rain_monthly["date"] <= df_t["date"].max())
].copy()

In [36]:
rain_monthly.head()
rain_monthly.groupby("region").size()

region
dalarna           127
gotland           127
jamtland          127
norrbotten        127
skane             127
stockholm         127
vasterbotten      127
vasternorrland    127
vastragotaland    127
dtype: int64

In [37]:
rain_monthly.groupby("region")["date"].agg(["min", "max"])

,min,max
region,,
dalarna,2015-07-01,2026-01-01
gotland,2015-07-01,2026-01-01
jamtland,2015-07-01,2026-01-01
norrbotten,2015-07-01,2026-01-01
skane,2015-07-01,2026-01-01
stockholm,2015-07-01,2026-01-01
vasterbotten,2015-07-01,2026-01-01
vasternorrland,2015-07-01,2026-01-01
vastragotaland,2015-07-01,2026-01-01


In [38]:
df_final = df_final.merge(
    rain_monthly,
    on=["region", "date"],
    how="left"
)

In [39]:
df_final.groupby("region")[["rain_sum", "rain_days"]].apply(lambda x: x.isna().sum())

,rain_sum,rain_days
region,,
dalarna,2,2
gotland,2,2
jamtland,2,2
norrbotten,2,2
riket,69,69
skane,2,2
stockholm,2,2
vasterbotten,2,2
vasternorrland,2,2


In [40]:
df_model = df_final.dropna().reset_index(drop=True)

In [66]:
df_model.sample(20)

,year,guest_nights,region,month,date,occupancy_rate,guest_lag1,guest_lag2,guest_lag12,occ_lag1,temp_mean,rain_sum,rain_days,rain_mean,guest_nights_total,occupancy_rate_total,share_of_total,occupancy_vs_total,seasonal_mean,above_seasonal_norm
9,2026,823702,dalarna,1,2026-01-01,0.485012,450619.0,125297.0,786074.0,0.372731,-6.6,67.2,14.0,2.167742,4260973.0,0.423378,0.193313,0.061634,7.359610e+05,1
574,2021,364618,stockholm,1,2021-01-01,0.217380,369181.0,408130.0,920746.0,0.218454,-1.2,87.6,12.0,2.825806,2226756.0,0.216977,0.163744,0.000403,8.920313e+05,0
802,2019,59227,vasternorrland,1,2019-01-01,0.390125,59675.0,68127.0,59382.0,0.382325,-6.6,23.3,7.0,0.751613,NaN,0.456810,NaN,-0.066685,5.804700e+04,1
355,2017,171653,norrbotten,2,2017-02-01,0.612130,134602.0,117621.0,172110.0,0.448867,-6.3,38.4,8.0,1.371429,NaN,0.528998,NaN,0.083132,NaN,0
53,2024,332976,dalarna,6,2024-06-01,0.440746,155363.0,273388.0,311065.0,0.329309,15.3,60.1,9.0,2.003333,7260255.0,0.573851,0.045863,-0.133105,2.878007e+05,1
265,2024,217602,jamtland,4,2024-04-01,0.405007,397838.0,421937.0,255410.0,0.541420,1.4,26.2,9.0,0.873333,3979292.0,0.506603,0.054684,-0.101596,2.823327e+05,0
439,2025,156401,norrbotten,10,2025-10-01,0.537538,187049.0,320780.0,142308.0,0.566856,6.2,88.0,15.0,2.838710,4444613.0,0.568252,0.035189,-0.030714,1.388163e+05,1
281,2022,156751,jamtland,6,2022-06-01,0.400846,105822.0,324416.0,131171.0,0.384734,13.7,52.8,10.0,1.760000,7072270.0,0.608768,0.022164,-0.207922,1.350433e+05,1
929,2021,219448,vastragotaland,2,2021-02-01,0.228146,178770.0,181378.0,476548.0,0.177292,-3.0,16.8,4.0,0.600000,2648072.0,0.275947,0.082871,-0.047801,4.517543e+05,0
237,2024,271803,jamtland,1,2024-01-01,0.391505,215035.0,78701.0,307323.0,0.374710,-7.4,30.1,9.0,0.970968,3903626.0,0.425283,0.069628,-0.033778,3.263547e+05,0


FÄRDIG!!!

In [42]:
riket_occ = pd.read_excel(
    "data/occupancy/riket.xlsx",
    skiprows=5,
    header=None
)

riket_occ.columns = ["month_name", "year", "occupancy_rate"]

C:\Users\ES7233\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [43]:
riket_occ["month"] = riket_occ["month_name"].map(month_map)

riket_occ["year"] = pd.to_numeric(riket_occ["year"], errors="coerce")
riket_occ["occupancy_rate"] = pd.to_numeric(
    riket_occ["occupancy_rate"],
    errors="coerce"
)

riket_occ = riket_occ.dropna(subset=["year", "month", "occupancy_rate"])

In [44]:
riket_occ["date"] = pd.to_datetime(
    dict(
        year=riket_occ["year"].astype(int),
        month=riket_occ["month"].astype(int),
        day=1
    )
)

In [45]:
riket_occ = (
    riket_occ[["date", "occupancy_rate"]]
    .rename(columns={"occupancy_rate": "occupancy_rate_total"})
    .sort_values("date")
    .reset_index(drop=True)
)

In [46]:
riket_gn = pd.read_excel(
    "data/guest_nights/riket.xlsx",
    skiprows=5,
    header=None
)

riket_gn.columns = ["month_name", "year", "guest_nights"]

C:\Users\ES7233\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [47]:
riket_gn["month"] = riket_gn["month_name"].map(month_map)

riket_gn["year"] = pd.to_numeric(riket_gn["year"], errors="coerce")
riket_gn["guest_nights"] = pd.to_numeric(
    riket_gn["guest_nights"],
    errors="coerce"
)

riket_gn = riket_gn.dropna(subset=["year", "month", "guest_nights"])

In [48]:
riket_gn["date"] = pd.to_datetime(
    dict(
        year=riket_gn["year"].astype(int),
        month=riket_gn["month"].astype(int),
        day=1
    )
)

In [49]:
riket_gn = (
    riket_gn[["date", "guest_nights"]]
    .rename(columns={"guest_nights": "guest_nights_total"})
    .sort_values("date")
    .reset_index(drop=True)
)

In [50]:
df_model = df_model.merge(
    riket_gn,
    on="date",
    how="left"
)

In [51]:
df_model = df_model.merge(
    riket_occ,
    on="date",
    how="left"
)

In [52]:
df_model["share_of_total"] = (
    df_model["guest_nights"] / df_model["guest_nights_total"]
)

df_model["occupancy_vs_total"] = (
    df_model["occupancy_rate"] - df_model["occupancy_rate_total"]
)

In [57]:
df_model.to_csv("sweden_tourism_weather.csv", index=False)

In [ ]:
# Logistisk regression START

In [97]:
# Låt oss försöka konstruera en binär variabel som vi kan göra logistisk regression på.
# above_seasonal_norm = 1 om guest_nights är högre än rullande 3-årsmedel

# Sortera så att rolling fungerar korrekt
df_model = df_model.sort_values(["region", "month", "year"]).reset_index(drop=True)

# Rullande 3-års medel för samma månad, baserat ENDAST på tidigare år
df_model["seasonal_mean"] = (
    df_model
    .groupby(["region", "month"])["guest_nights"]
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)

# Skapa binär target
df_model["above_seasonal_norm"] = (
    df_model["guest_nights"] > df_model["seasonal_mean"]
).astype(int)

# Droppa rader utan historik (första observationen per region+månad)
df_model_clf = df_model.dropna(subset=["seasonal_mean"]).reset_index(drop=True)

# Sortera tillbaka till kronologisk ordning
df_model_clf = df_model_clf.sort_values(["region", "date"]).reset_index(drop=True)

In [98]:
# Kolla balansen — logistisk regression vill helst ha någorlunda balanserade klasser
print(df_model_clf["above_seasonal_norm"].value_counts(normalize=True))

# Per region — finns det regioner där nästan allt är 0 eller 1?
print(df_model_clf.groupby("region")["above_seasonal_norm"].mean())

above_seasonal_norm
1    0.646739
0    0.353261
Name: proportion, dtype: float64
region
dalarna           0.553398
gotland           0.524272
jamtland          0.466019
norrbotten        0.766990
skane             0.744898
stockholm         0.766990
vasterbotten      0.592233
vasternorrland    0.669903
vastragotaland    0.742574
Name: above_seasonal_norm, dtype: float64


In [99]:
import numpy as np

# Cyklisk månadskodning
df_model_clf["month_sin"] = np.sin(2 * np.pi * df_model_clf["month"] / 12)
df_model_clf["month_cos"] = np.cos(2 * np.pi * df_model_clf["month"] / 12)

# Spara originalkolumnen INNAN get_dummies
df_model_clf["regions_original"] = df_model_clf["region"]

# One-hot encoding av region
df_model_clf = pd.get_dummies(
    df_model_clf,
    columns=["region"],
    prefix="region",
    drop_first=True,   # undvik dummy-fällan
    dtype=int          # int istället för bool — viktigt för statsmodels
)

In [100]:
feature_cols = [
    # Väder
    "temp_mean", "rain_sum", "rain_days",
    # Säsong
    "month_sin", "month_cos",
    # Historik
    "guest_lag1", "guest_lag12", "occ_lag1",
    # Region (alla region_-kolumner från one-hot)
] + [c for c in df_model_clf.columns if c.startswith("region_")]

X = df_model_clf[feature_cols]
y = df_model_clf["above_seasonal_norm"]

In [101]:
# Välj ett brytdatum — t.ex. sista 20% av tidsperioden som test
split_date = df_model_clf["date"].quantile(0.8)

train_mask = df_model_clf["date"] < split_date
test_mask = df_model_clf["date"] >= split_date

X_train = X[train_mask]
y_train = y[train_mask]
X_test = X[test_mask]
y_test = y[test_mask]

print(f"Train: {len(X_train)} observationer ({df_model_clf.loc[train_mask, 'date'].min().date()} → {df_model_clf.loc[train_mask, 'date'].max().date()})")
print(f"Test:  {len(X_test)} observationer ({df_model_clf.loc[test_mask, 'date'].min().date()} → {df_model_clf.loc[test_mask, 'date'].max().date()})")
print(f"\nKlassbalans train: {y_train.mean():.3f}")
print(f"Klassbalans test:  {y_test.mean():.3f}")

Train: 729 observationer (2017-07-01 → 2024-03-01)
Test:  191 observationer (2024-04-01 → 2026-01-01)

Klassbalans train: 0.658
Klassbalans test:  0.602


In [102]:
from sklearn.preprocessing import StandardScaler

# Standardisera ENDAST de numeriska features — region-dummies lämnas i fred
numeric_features = ["temp_mean", "rain_sum", "rain_days",
                    "month_sin", "month_cos",
                    "guest_lag1", "guest_lag12", "occ_lag1"]

scaler = StandardScaler()

# Fit ENDAST på träningsdata, transform på båda
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test_scaled[numeric_features] = scaler.transform(X_test[numeric_features])

In [103]:
import statsmodels.api as sm

# statsmodels — för tolkning
X_train_sm = X_train_scaled.copy()
X_train_sm.insert(0, "const", 1.0)

logit_model = sm.Logit(y_train, X_train_sm).fit()
print(logit_model.summary())

Optimization terminated successfully.
         Current function value: 0.521011
         Iterations 6
                            Logit Regression Results                           
Dep. Variable:     above_seasonal_norm   No. Observations:                  729
Model:                           Logit   Df Residuals:                      712
Method:                            MLE   Df Model:                           16
Date:                 Tue, 05 May 2026   Pseudo R-squ.:                  0.1885
Time:                         21:33:04   Log-Likelihood:                -379.82
converged:                        True   LL-Null:                       -468.07
Covariance Type:             nonrobust   LLR p-value:                 4.231e-29
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                     0.9223      0.261      3.537      0.000       0.411 

In [ ]:
# occ_lag1 bär föregående månads beläggning.§
# Det visar sig att vädret inte verkar ha statistisk signifikans, utan det som mest förklarar signalen för om en månad slår säsongsnormen är vad som hände månaden innan. 
# Om juni var en stark månad i Stockholm så är sannolikheten väldigt hög att juli också gör det.

In [ ]:
# Provar att köra om utan occ_lag1 för att se om vädervariablerna blir bättre

In [85]:
feature_cols_b = [
    # Väder
    "temp_mean", "rain_sum", "rain_days",
    # Säsong
    "month_sin", "month_cos",
    # Historik — endast långsiktig (samma månad föregående år)
    "guest_lag12",
    # Region
] + [c for c in df_model_clf.columns 
     if c.startswith("region_")]

In [86]:
X_b = df_model_clf[feature_cols_b]
y_b = df_model_clf["above_seasonal_norm"]

# Samma kronologiska split som tidigare
split_date = df_model_clf["date"].quantile(0.8)
train_mask = df_model_clf["date"] < split_date
test_mask = df_model_clf["date"] >= split_date

X_train_b = X_b[train_mask]
y_train_b = y_b[train_mask]
X_test_b = X_b[test_mask]
y_test_b = y_b[test_mask]

# Standardisering — endast numeriska features
numeric_features_b = ["temp_mean", "rain_sum", "rain_days",
                      "month_sin", "month_cos", "guest_lag12"]

from sklearn.preprocessing import StandardScaler
scaler_b = StandardScaler()

X_train_scaled_b = X_train_b.copy()
X_test_scaled_b = X_test_b.copy()

X_train_scaled_b[numeric_features_b] = scaler_b.fit_transform(X_train_b[numeric_features_b])
X_test_scaled_b[numeric_features_b] = scaler_b.transform(X_test_b[numeric_features_b])

In [87]:
import statsmodels.api as sm

X_train_sm_b = X_train_scaled_b.copy()
X_train_sm_b.insert(0, "const", 1.0)

logit_model_b = sm.Logit(y_train_b, X_train_sm_b).fit()
print(logit_model_b.summary())

Optimization terminated successfully.
         Current function value: 0.622044
         Iterations 5
                            Logit Regression Results                           
Dep. Variable:     above_seasonal_norm   No. Observations:                  729
Model:                           Logit   Df Residuals:                      714
Method:                            MLE   Df Model:                           14
Date:                 Tue, 05 May 2026   Pseudo R-squ.:                 0.03119
Time:                         09:28:38   Log-Likelihood:                -453.47
converged:                        True   LL-Null:                       -468.07
Covariance Type:             nonrobust   LLR p-value:                  0.009836
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                     0.3004      0.228      1.319      0.187      -0.146 

In [ ]:
# Här saknar modellen nästan helt förklaringskraft, Pseudo R-squ.: 0.03119. 
# Vi undersökte om vädervariabler kan förutsäga turistmönster. Vi fann att den enda starka prediktorn var autokorrelation, och att rena vädervariabler bidrar med ytterst lite förklaringskraft

In [ ]:
# Logistisk regression SLUT


# KNN START

In [88]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# Börja med en standardstart: k = sqrt(n) ≈ 27 för 729 observationer
# Vi testar några värden för att se hur k påverkar
for k in [3, 5, 11, 21, 31]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled_b, y_train_b)
    
    train_acc = accuracy_score(y_train_b, knn.predict(X_train_scaled_b))
    test_acc = accuracy_score(y_test_b, knn.predict(X_test_scaled_b))
    test_auc = roc_auc_score(y_test_b, knn.predict_proba(X_test_scaled_b)[:, 1])
    
    print(f"k={k:2d}  train_acc={train_acc:.3f}  test_acc={test_acc:.3f}  test_auc={test_auc:.3f}")

k= 3  train_acc=0.759  test_acc=0.618  test_auc=0.651
k= 5  train_acc=0.700  test_acc=0.634  test_auc=0.674
k=11  train_acc=0.686  test_acc=0.623  test_auc=0.648
k=21  train_acc=0.680  test_acc=0.602  test_auc=0.654
k=31  train_acc=0.680  test_acc=0.628  test_auc=0.671


In [89]:
# Logistisk regression Modell B på testdata
X_test_sm_b = X_test_scaled_b.copy()
X_test_sm_b.insert(0, "const", 1.0)

y_pred_proba_b = logit_model_b.predict(X_test_sm_b)
y_pred_b = (y_pred_proba_b >= 0.5).astype(int)

print(f"\nLogit B test_acc={accuracy_score(y_test_b, y_pred_b):.3f}  test_auc={roc_auc_score(y_test_b, y_pred_proba_b):.3f}")


Logit B test_acc=0.634  test_auc=0.721


In [ ]:
# Vi undersökte om väder förutsäger turismavvikelser i svenska regioner. 
# Den dominerande signalen i datan är autokorrelation — föregående månads beläggning. 
# Detta säger något om svensk turism: den är trögare än den är väderkänslig på månadsnivå

In [90]:
X_test_sm_a = X_test_scaled.copy()
X_test_sm_a.insert(0, "const", 1.0)

y_pred_proba_a = logit_model.predict(X_test_sm_a)
y_pred_a = (y_pred_proba_a >= 0.5).astype(int)

print(f"Logit A test_acc={accuracy_score(y_test, y_pred_a):.3f}  test_auc={roc_auc_score(y_test, y_pred_proba_a):.3f}")

Logit A test_acc=0.623  test_auc=0.597


In [92]:
print("Confusion matrix (Logit A):")
print(confusion_matrix(y_test, y_pred_a))
print()
print(classification_report(y_test, y_pred_a, target_names=['under norm', 'över norm']))

Confusion matrix (Logit A):
[[22 54]
 [18 97]]

              precision    recall  f1-score   support

  under norm       0.55      0.29      0.38        76
   över norm       0.64      0.84      0.73       115

    accuracy                           0.62       191
   macro avg       0.60      0.57      0.55       191
weighted avg       0.61      0.62      0.59       191



In [91]:
from sklearn.metrics import confusion_matrix, classification_report

print("Confusion matrix (Logit B):")
print(confusion_matrix(y_test_b, y_pred_b))
print()
print(classification_report(y_test_b, y_pred_b, target_names=['under norm', 'över norm']))

Confusion matrix (Logit B):
[[ 13  63]
 [  7 108]]

              precision    recall  f1-score   support

  under norm       0.65      0.17      0.27        76
   över norm       0.63      0.94      0.76       115

    accuracy                           0.63       191
   macro avg       0.64      0.56      0.51       191
weighted avg       0.64      0.63      0.56       191



In [ ]:
# Mycket svagt resultat